# EDA Skeleton — Columbia RealTalk + FLAME-derived backchannel labels

Set `REALTALK_ROOT` to your local RealTalk mount, then run all cells.
Until data is available, synthetic demo arrays illustrate the plots you will produce.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REALTALK_ROOT = Path(os.environ.get("REALTALK_ROOT", "")).expanduser()
USE_SYNTHETIC = not REALTALK_ROOT.exists() or not any(REALTALK_ROOT.iterdir()) if REALTALK_ROOT.exists() else True
print("REALTALK_ROOT:", REALTALK_ROOT or "(unset)")
print("Mode:", "SYNTHETIC DEMO" if USE_SYNTHETIC else "REAL DATA")

CLASSES = [
    "nod", "shake", "tilt", "lean_forward", "lean_back", "eyebrow_raise", "neutral"
]

## 1. Inventory helpers

Adapt `list_videos` / `load_flame` to your on-disk layout (npz records, NPZ per clip, etc.).

In [ ]:
def list_videos(root: Path):
    """Return list of clip identifiers under RealTalk root."""
    if not root.exists():
        return []
    # TODO: adjust glob to actual RealTalk layout
    patterns = ["**/*.mp4", "**/*.avi", "**/video.*"]
    files = []
    for p in patterns:
        files.extend(root.glob(p))
    return sorted(set(files))


def load_flame_stub(clip_id, n_frames=250, fps=25):
    """Placeholder FLAME loader: pitch/yaw/roll + translation + brow proxy."""
    t = np.arange(n_frames) / fps
    rng = np.random.default_rng(abs(hash(str(clip_id))) % (2**32))
    pitch = 0.05 * np.sin(2 * np.pi * 2.0 * t) * (rng.random() > 0.4)
    yaw = 0.04 * np.sin(2 * np.pi * 1.5 * t + 0.3) * (rng.random() > 0.7)
    roll = 0.03 * np.sin(2 * np.pi * 1.2 * t) * (rng.random() > 0.75)
    trans_z = 0.01 * np.cumsum(rng.normal(0, 1, size=n_frames))
    brow = np.clip(rng.normal(0, 0.2, size=n_frames), 0, None)
    return {
        "fps": fps,
        "pitch": pitch,
        "yaw": yaw,
        "roll": roll,
        "trans_z": trans_z,
        "brow": brow,
    }


videos = list_videos(REALTALK_ROOT)
print(f"Found {len(videos)} video files")
if USE_SYNTHETIC:
    videos = [f"synth_clip_{i:03d}" for i in range(20)]
    print("Using 20 synthetic clips for demo plots")

## 2. Label rules (nod band-pass sketch)

Mirrors `api/label_rules.py` used by the demo website.

In [ ]:
from scipy.signal import butter, filtfilt, find_peaks


def bandpass(x, fps, low=1.0, high=3.0, order=3):
    if len(x) < 16:
        return np.zeros_like(x)
    nyq = 0.5 * fps
    b, a = butter(order, [low / nyq, high / nyq], btype="band")
    return filtfilt(b, a, x)


def detect_nod(pitch, fps=25, min_cycles=2):
    filtered = bandpass(pitch, fps)
    peaks, _ = find_peaks(filtered, height=np.std(filtered) * 0.5)
    return len(peaks) >= min_cycles, filtered, peaks


def label_window(flame: dict) -> str:
    """Single-label priority: nod > shake > eyebrow > tilt > lean_f > lean_b > neutral."""
    fps = flame["fps"]
    nod, _, _ = detect_nod(flame["pitch"], fps=fps)
    if nod:
        return "nod"
    yaw_f = bandpass(flame["yaw"], fps, 0.8, 2.5)
    if np.max(np.abs(yaw_f)) > 0.02 and len(find_peaks(np.abs(yaw_f))[0]) >= 2:
        return "shake"
    if np.mean(flame["brow"]) > 0.35:
        return "eyebrow_raise"
    if np.max(np.abs(flame["roll"])) > 0.05:
        return "tilt"
    dz = flame["trans_z"][-1] - flame["trans_z"][0]
    if dz > 0.05:
        return "lean_forward"
    if dz < -0.05:
        return "lean_back"
    return "neutral"

## 3. Class histogram

In [ ]:
labels = []
for clip in videos:
    flame = load_flame_stub(clip)
    # In real data: sliding windows over the full sequence
    for start in range(0, len(flame["pitch"]) - 25, 25):
        window = {k: (v[start:start+50] if isinstance(v, np.ndarray) else v) for k, v in flame.items()}
        if isinstance(window["pitch"], np.ndarray) and len(window["pitch"]) < 16:
            continue
        labels.append(label_window(window))

counts = pd.Series(labels).value_counts().reindex(CLASSES, fill_value=0)
display(counts.to_frame("count"))

fig, ax = plt.subplots(figsize=(8, 4))
counts.plot(kind="bar", ax=ax, color="#1f4e5f")
ax.set_title("Derived backchannel class counts (demo or real)")
ax.set_ylabel("windows")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 4. Example nod detection plot

In [ ]:
flame = load_flame_stub("demo_nod", n_frames=125)
# Force a clear nod-like pitch for illustration
t = np.arange(125) / 25
flame["pitch"] = 0.08 * np.sin(2 * np.pi * 2.0 * t)
ok, filtered, peaks = detect_nod(flame["pitch"], fps=25)

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(t, flame["pitch"], label="pitch", alpha=0.5)
ax.plot(t, filtered, label="1–3 Hz band-pass")
ax.scatter(t[peaks], filtered[peaks], color="crimson", zorder=3, label="peaks")
ax.set_title(f"Nod detector demo — fired={ok}")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Export summary for dissertation tables

After running on real data, save CSV counts for Chapter figures.

In [ ]:
out = Path("../docs/eda_class_counts.csv")
counts.to_csv(out, header=["count"])
print("Wrote", out.resolve())